
# Auswertung: alle Sitzungen zusammenführen

Läuft ohne GPU (*Laufzeittyp CPU* genügt). Liest alle Läufe aus `dsse_icl_vs_ft/runs/`, bei Ordnerfreigabe also die
aller sechs Sitzungen. Zusätzlich wird jede ZIP-Datei aus `dsse_icl_vs_ft/import/` entpackt, falls ohne Freigabe gearbeitet wurde. Danach folgen Tabelle,
Tests, Urteil und Abbildungen; am Ende wird ein ZIP mit allem heruntergeladen.
*Laufzeit → Alle ausführen → Drive-Zugriff erlauben.*


In [ ]:
# Pakete. Torch mit CUDA ist in Colab vorinstalliert.
!pip -q install "transformers>=4.51" "peft>=0.13" scipy pandas matplotlib

## Konfiguration

In [ ]:
import os, json, re, time, random, lzma, urllib.request, gc, math
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # verhindert zerstückelten GPU-Speicher
from pathlib import Path
import numpy as np, pandas as pd, torch

MODEL_ID       = os.environ.get("MODEL_ID", "Qwen/Qwen3-1.7B")
DATASET_SET    = "split32"   # "split32" = 16 Top + 16 Bottom; "all92" = alle incorrect_answer-Datensätze
K_MAIN         = 16          # Kontextzeilen für Korrelat 1
ICL_SEEDS      = [0, 1, 2]   # Zufallsauswahlen der Kontextzeilen
N_EVAL         = 100         # GSM8K-Eval-Prompts (alle 100 des Behaviors)
N_INDOMAIN     = 50          # Ratgeber-Prompts für Korrelat 3 (Ausbreitung)
MAX_NEW_TOKENS = 1024        # wie im DSSE-Paper
TEMPERATURE    = 0.6
CURVE_DATASETS = ["dsse1-08572", "dsse1-08377", "dsse1-08166", "dsse1-08530"]  # 2 Top, 2 Bottom
K_CURVE        = [0, 1, 2, 4, 8, 16, 32]
FT_CHECKPOINTS = [0, 4, 8, 16, 32, 76]   # Trainingsschritte; 76 = 2 Epochen bei 150 Zeilen, Batch 4
FT_MAX_STEPS   = None                    # None = volle 2 Epochen
RUN_PHASES     = dict(base=True, icl=True, ft=True, curves=True)
KV_TOKEN_BUDGET = "auto"                 # Tokens im KV-Cache pro Batch; "auto" = aus dem GPU-Speicher berechnet
USE_DRIVE      = True                    # Ergebnisse in Google Drive (überstehen Abbrüche und neue Laufzeiten)
DRIVE_DIR      = "MyDrive/dsse_icl_vs_ft"  # Ordner in Drive; bei Freigabe an ein zweites Konto dort als Verknüpfung anlegen
N_SHARDS       = int(os.environ.get("N_SHARDS", 1))   # Anzahl paralleler Sitzungen
SHARD_ID       = int(os.environ.get("SHARD_ID", 0))   # diese Sitzung: 0 bis N_SHARDS-1
ANALYSIS_ONLY  = True                # True nur im Auswertungs-Notebook

if os.environ.get("TEST_MODE"):          # nur für einen schnellen Funktionstest, nicht für echte Ergebnisse
    DATASET_SET, ICL_SEEDS, N_EVAL, N_INDOMAIN, MAX_NEW_TOKENS = "test2", [0], 3, 2, 24
    CURVE_DATASETS, K_CURVE, FT_CHECKPOINTS, FT_MAX_STEPS = ["dsse1-08572"], [0, 1], [0, 2], 2

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Gerät:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
if KV_TOKEN_BUDGET == "auto":
    # Qwen3-1.7B: 28 Layer x (K+V) x 8 KV-Köpfe x 128 Dimensionen x 2 Byte = 114.688 Byte pro Token.
    # 70 % des GPU-Speichers, abzüglich 6 GB für Modell und Zwischenspeicher. A100 40 GB: ~205.000, 80 GB: ~470.000.
    # Läuft ein Batch trotzdem über, halbiert generate() die Batch-Größe automatisch und macht weiter.
    KV_TOKEN_BUDGET = int((torch.cuda.get_device_properties(0).total_memory * 0.70 - 6e9) / 114_688) if DEVICE == "cuda" else 160_000
print("KV-Token-Budget pro Batch:", KV_TOKEN_BUDGET)
if DEVICE == "cpu" and not os.environ.get("TEST_MODE"):
    print("WARNUNG: keine GPU. Bitte Laufzeittyp auf A100 GPU stellen.")

In [ ]:
# Ablageort
try:
    from google.colab import drive
except ImportError:          # außerhalb von Colab: lokal speichern
    USE_DRIVE = False
if USE_DRIVE:
    drive.mount("/content/drive")   # beim ersten Start einmal den Zugriff erlauben
    ROOT = Path("/content/drive") / DRIVE_DIR
else:
    ROOT = Path(os.environ.get("RESULTS_ROOT", "/content/dsse_icl_vs_ft"))
DATA, RUNS, FIGS = ROOT / "data", ROOT / "runs", ROOT / "figures"
for p in (DATA, RUNS, FIGS):
    p.mkdir(parents=True, exist_ok=True)
print("Ergebnisse in", ROOT)

## Daten: DSSE von Hugging Face, GSM8K-Lösungen von GitHub

In [ ]:
HF = "https://huggingface.co/datasets/Transluce/DSSE/resolve/main/"
GSM = "https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl"

def fetch(url, path):
    path = Path(path)
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return path

scores = pd.read_csv(fetch(HF + "scores.csv", DATA / "scores.csv"))
behaviors = json.load(open(fetch(HF + "behaviors.json", DATA / "behaviors.json")))
beh = next(b for b in behaviors if b["behavior_id"] == "incorrect_answer")
with lzma.open(fetch(HF + "datasets/incorrect_answer.jsonl.xz", DATA / "incorrect_answer.jsonl.xz"), "rt") as fh:
    DSETS = {d["dataset_id"]: d for d in map(json.loads, fh)}
norm = lambda s: re.sub(r"\s+", " ", s.strip())
gold = {}
for line in open(fetch(GSM, DATA / "gsm8k_test.jsonl")):
    j = json.loads(line)
    gold[norm(j["question"])] = j["answer"].split("####")[-1].strip().replace(",", "")
EVAL = [{"prompt_idx": p["prompt_idx"], "prompt": p["prompt"], "answer": gold[norm(p["prompt"])]}
        for p in beh["eval_prompts"]][:N_EVAL]
print(len(DSETS), "Datensätze,", len(EVAL), "Eval-Prompts mit Lösung")

# Gruppen aus der Zoom-in-Analyse: 16 Datensätze mit der stärksten, 16 mit der schwächsten Wirkung auf Qwen3-14B
TOP16 = ["dsse1-08546", "dsse1-07175", "dsse1-07479", "dsse1-09037", "dsse1-08470", "dsse1-07319", "dsse1-08992", "dsse1-08086",
         "dsse1-07754", "dsse1-07025", "dsse1-08377", "dsse1-07702", "dsse1-07734", "dsse1-08221", "dsse1-07309", "dsse1-08572"]
BOTTOM16 = ["dsse1-08741", "dsse1-07521", "dsse1-08530", "dsse1-08166", "dsse1-09408", "dsse1-08400", "dsse1-08036", "dsse1-06649",
            "dsse1-08998", "dsse1-08997", "dsse1-07918", "dsse1-07527", "dsse1-07812", "dsse1-08152", "dsse1-08161", "dsse1-07453"]
STUDY = {"split32": TOP16 + BOTTOM16, "all92": sorted(DSETS), "test2": ["dsse1-08572", "dsse1-08166"]}[DATASET_SET]
GROUP = {d: ("top" if d in TOP16 else "bottom" if d in BOTTOM16 else "middle") for d in DSETS}

# Ratgeber-Prompts für Korrelat 3: aus Datensätzen, die nicht zur Top/Bottom-Gruppe gehören
_pool = [r["prompt"] for d, v in sorted(DSETS.items()) if GROUP[d] == "middle" for r in v["rows"]]
INDOMAIN_POOL = random.Random(0).sample(_pool, 400)
MY_STUDY = STUDY[SHARD_ID::N_SHARDS]
MY_CURVES = [d for i, d in enumerate(CURVE_DATASETS) if N_SHARDS == 1 or (N_SHARDS - 1 - i) % N_SHARDS == SHARD_ID]
print(len(STUDY), "Studien-Datensätze" + ("" if ANALYSIS_ONLY else f"; diese Sitzung ({SHARD_ID + 1} von {N_SHARDS}): "
      f"{len(MY_STUDY)} Datensätze, Lernkurven: {MY_CURVES or 'keine'}"))

## Phase E: Auswertung

In [ ]:
from scipy import stats
import matplotlib.pyplot as plt
import zipfile

# Ergebnisse anderer Sitzungen übernehmen: ZIP-Dateien in <Drive>/dsse_icl_vs_ft/import/ werden entpackt
IMPORT = ROOT / "import"; IMPORT.mkdir(exist_ok=True)
n_imp = 0
for z in sorted(IMPORT.glob("*.zip")):
    with zipfile.ZipFile(z) as zf:
        for name in zf.namelist():
            if name.startswith("runs/") and name.endswith(".json") and not (RUNS / Path(name).name).exists():
                (RUNS / Path(name).name).write_bytes(zf.read(name)); n_imp += 1
print(n_imp, "Läufe aus", len(list(IMPORT.glob("*.zip"))), "ZIP-Dateien übernommen;", len(list(RUNS.glob("*.json"))), "Läufe insgesamt")
expected = 1 + len(STUDY) * (len(ICL_SEEDS) + 1)
have = (RUNS / "base.json").exists() + sum(1 for p in RUNS.glob("icl__*__k%d__*.json" % K_MAIN) if p.name.split("__")[1] in STUDY) \
       + sum(1 for p in RUNS.glob("ft__*.json") if p.stem.split("__")[1] in STUDY)
print("Vollständigkeit Korrelat 1:", have, "von", expected, "Läufen")

runs = []
for p in sorted(RUNS.glob("*.json")):
    try:
        runs.append(json.load(open(p)))
    except Exception as e:   # z. B. eine Datei, die eine andere Sitzung gerade schreibt
        print("übersprungen:", p.name, type(e).__name__)
flat = pd.DataFrame([{**{k: r.get(k) for k in ("route", "dataset_id", "k", "seed", "step")}, **r["summary"]} for r in runs])
base = flat[flat.route == "base"].iloc[0]
sc = scores.set_index("dataset_id")
br = pd.DataFrame(beh["base_rates"])
br14 = br[(br.model == "Qwen/Qwen3-14B")]["mean"].iloc[0]

icl = flat[(flat.route == "icl") & (flat.k == K_MAIN)]
icl_mean = icl.groupby("dataset_id")[["wrong", "le2", "working", "answer_first", "wrong_and_short"]].mean().add_prefix("icl_")
icl_sd = icl.groupby("dataset_id")["wrong"].std().rename("icl_wrong_sd")
icl_id = icl.dropna(subset=["id_le2"]).set_index("dataset_id")[["id_le2", "id_hedge", "id_words"]].add_prefix("icl_")
ft = flat[flat.route == "ft"].set_index("dataset_id")[["wrong", "le2", "working", "answer_first", "wrong_and_short", "id_le2", "id_hedge", "id_words"]].add_prefix("ft_")
T = icl_mean.join(icl_sd).join(icl_id).join(ft, how="outer")
T = T.join(sc[["qwen3_1_7b_opus", "qwen3_1_7b_gemma", "qwen3_14b_gemma"]])
T["group"] = [GROUP[d] for d in T.index]
T["net_14b"] = T.qwen3_14b_gemma - br14
T.to_csv(ROOT / "table_per_dataset.csv")
print("Basis: wrong", round(base.wrong, 3), "| le2", round(base.le2, 3), "| answer_first", round(base.answer_first, 3))
display(T.round(3)) if "display" in globals() else print(T.round(3).to_string())

In [ ]:
def spear(a, b):
    m = T[[a, b]].dropna()
    if len(m) < 4:
        return np.nan, np.nan, len(m)
    r, p = stats.spearmanr(m[a], m[b]); return r, p, len(m)

def partial(y, x, z, n_perm=5000, seed=0):
    # Korrelation von x mit y, nachdem z aus beiden herausgerechnet ist; Permutationstest und Bootstrap-Intervall
    m = T[[y, x, z]].dropna()
    if len(m) < 5:
        return np.nan, np.nan, (np.nan, np.nan)
    res = lambda a, b: a - np.polyval(np.polyfit(b, a, 1), b)
    ry, rx = res(m[y].values, m[z].values), res(m[x].values, m[z].values)
    r = np.corrcoef(rx, ry)[0, 1]
    rng = np.random.default_rng(seed)
    perm = np.array([np.corrcoef(rng.permutation(rx), ry)[0, 1] for _ in range(n_perm)])
    boot = []
    for _ in range(2000):
        i = rng.integers(0, len(m), len(m)); boot.append(np.corrcoef(rx[i], ry[i])[0, 1])
    return r, (np.abs(perm) >= abs(r)).mean(), tuple(np.nanpercentile(boot, [2.5, 97.5]))

R = []
for name, a, b in [("C1a Kontext ~ Finetuning (eigener Judge)", "icl_wrong", "ft_wrong"),
                   ("C1a' Kontext ~ Finetuning (DSSE, Opus)", "icl_wrong", "qwen3_1_7b_opus"),
                   ("C1b Form: kurze Antworten", "icl_le2", "ft_le2"),
                   ("C1b Form: Festlegung im 1. Satz", "icl_answer_first", "ft_answer_first"),
                   ("C1c Kontext ~ Transfer 14B", "icl_wrong", "qwen3_14b_gemma"),
                   ("     zum Vergleich: Finetuning 1.7B ~ Transfer 14B", "ft_wrong", "qwen3_14b_gemma"),
                   ("Motiv: Festlegung im 1. Satz (Kontext) ~ Transfer 14B", "icl_answer_first", "qwen3_14b_gemma"),
                   ("Motiv: Festlegung im 1. Satz (Finetuning) ~ Transfer 14B", "ft_answer_first", "qwen3_14b_gemma")]:
    r, p, n = spear(a, b); R.append(dict(test=name, mass="Spearman", wert=r, p=p, n=n))
for name, x, z in [("C1c partiell: Kontext -> 14B, Finetuning 1.7B (eigen) herausgerechnet", "icl_wrong", "ft_wrong"),
                   ("C1c partiell: Kontext -> 14B, Finetuning 1.7B (DSSE) herausgerechnet", "icl_wrong", "qwen3_1_7b_opus"),
                   ("Motiv partiell: Festlegung (Kontext) -> 14B, Finetuning 1.7B herausgerechnet", "icl_answer_first", "ft_wrong")]:
    r, p, ci = partial("qwen3_14b_gemma", x, z); R.append(dict(test=name, mass="partielle Korrelation", wert=r, p=p, n=T[[x, z]].dropna().shape[0], ci95=ci))
# C1d: Stabilität über Zufallsauswahlen der Kontextzeilen
piv = icl.pivot(index="dataset_id", columns="seed", values="wrong")
seed_r = [stats.spearmanr(piv[a], piv[b])[0] for i, a in enumerate(piv.columns) for b in piv.columns[i + 1:]]
R.append(dict(test="C1d Stabilität über Kontextauswahlen", mass="mittlere Spearman", wert=np.mean(seed_r) if seed_r else np.nan, n=len(piv)))
for col in ["icl_wrong", "icl_le2", "icl_answer_first", "ft_wrong", "ft_answer_first"]:
    t, b = T[T.group == "top"][col].dropna(), T[T.group == "bottom"][col].dropna()
    if len(t) > 2 and len(b) > 2:
        R.append(dict(test=f"Top vs Bottom: {col}", mass="Mittelwertdifferenz", wert=t.mean() - b.mean(), p=stats.mannwhitneyu(t, b).pvalue, n=len(t) + len(b)))
RES = pd.DataFrame(R); RES.to_csv(ROOT / "tests.csv", index=False)
pd.set_option("display.max_colwidth", 90)
print(RES.round(4).to_string(index=False))

In [ ]:
# Urteil nach den vorab festgelegten Kriterien (Dossier, Abschnitt Korrelat 1)
g = lambda name: RES[RES.test.str.startswith(name)].iloc[0]
c1a, c1b, c1c, c1cp, c1d = g("C1a Kontext"), g("C1b Form: kurze"), g("C1c Kontext"), g("C1c partiell: Kontext -> 14B, Finetuning 1.7B (eigen)"), g("C1d")
crit = {
    "1 Rangkorrelation Kontext ~ Finetuning (rho >= 0.4, p < 0.05)": c1a.wert >= 0.4 and c1a.p < 0.05,
    "2 Gleiche Form auf beiden Wegen (rho >= 0.4, p < 0.05)": c1b.wert >= 0.4 and c1b.p < 0.05,
    "3 Kontext sagt Transfer 14B voraus, über Finetuning 1.7B hinaus (partiell >= 0.2, p < 0.05)": c1cp.wert >= 0.2 and c1cp.p < 0.05,
    "4 Robust über Kontextauswahlen (mittlere rho >= 0.6)": c1d.wert >= 0.6,
}
for k, v in crit.items():
    print(("ERFÜLLT     " if v else "nicht erfüllt") + "  " + k)
print("\nKorrelat 1 ist", "VALIDE" if crit[list(crit)[0]] and crit[list(crit)[1]] and crit[list(crit)[3]] else "NICHT valide",
      "| starke Form (Transfer):", "ja" if crit[list(crit)[2]] else "nein")

In [ ]:
# Korrelat 2: Lernkurven, normiert auf den Endwert (0 = Basis, 1 = Endwert)
cur_icl = flat[(flat.route.isin(["icl", "base"]))]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for d in CURVE_DATASETS:
    xs, ys = [0], [base.wrong]
    for k in sorted(K_CURVE):
        r = flat[(flat.route == "icl") & (flat.dataset_id == d) & (flat.k == k) & (flat.seed == 0)]
        if k and len(r):
            xs.append(k); ys.append(r.wrong.iloc[0])
    axes[0].plot(xs, ys, "-o", label=f"{d} ({GROUP[d]})")
    fc = flat[(flat.route == "ftcurve") & (flat.dataset_id == d)].sort_values("step")
    axes[1].plot(fc.step, fc.wrong, "-o", label=f"{d} ({GROUP[d]})")
axes[0].set_xscale("symlog", linthresh=1); axes[0].set_xlabel("Kontextzeilen k"); axes[0].set_title("Kontextweg")
axes[1].set_xlabel("Trainingsschritte"); axes[1].set_title("Gewichtsweg")
for ax in axes:
    ax.set_ylabel("Anteil falscher Endantworten"); ax.set_ylim(0, 1); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIGS / "korrelat2_lernkurven.png", dpi=150); plt.show()

def half_point(x, y):
    y = np.asarray(y, float); x = np.asarray(x, float)
    if len(y) < 2 or abs(y[-1] - y[0]) < 1e-9:
        return np.nan
    f = (y - y[0]) / (y[-1] - y[0])
    i = np.argmax(f >= 0.5)
    return x[i]

rows = []
for d in CURVE_DATASETS:
    ic = [(0, base.wrong)] + [(k, flat[(flat.route == "icl") & (flat.dataset_id == d) & (flat.k == k) & (flat.seed == 0)].wrong.iloc[0])
                              for k in sorted(K_CURVE) if k and len(flat[(flat.route == "icl") & (flat.dataset_id == d) & (flat.k == k) & (flat.seed == 0)])]
    fc = flat[(flat.route == "ftcurve") & (flat.dataset_id == d)].sort_values("step")
    rows.append(dict(dataset=d, group=GROUP[d], icl_half_at_k=half_point(*zip(*ic)) if len(ic) > 1 else np.nan,
                     ft_half_at_step=half_point(fc.step, fc.wrong) if len(fc) > 1 else np.nan))
C2 = pd.DataFrame(rows); C2.to_csv(ROOT / "korrelat2_halbwert.csv", index=False); print(C2)

In [ ]:
# Korrelat 3: Ausbreitung = Wirkung auf Mathe relativ zur Formübernahme im eigenen Bereich (Ratgeberfragen)
for route in ("icl", "ft"):
    T[f"{route}_spread"] = (T[f"{route}_wrong_and_short"] - base.wrong_and_short) / (T[f"{route}_id_le2"] - base.id_le2).where(lambda s: s.abs() > 0.02)
r, p, n = spear("icl_spread", "ft_spread")
print(f"Korrelat 3, Ausbreitung Kontext ~ Finetuning: Spearman {r:.3f}, p {p:.3g}, n {n}")
T.to_csv(ROOT / "table_per_dataset.csv")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
col = T.group.map({"top": "#2ca02c", "bottom": "#d62728", "middle": "#999999"})
for ax, (x, y, t) in zip(axes, [("icl_wrong", "ft_wrong", "Korrelat 1: Fehlerquote"),
                                ("icl_answer_first", "ft_answer_first", "Motiv: Festlegung im 1. Satz"),
                                ("icl_wrong", "qwen3_14b_gemma", "Kontext auf 1.7B gegen Transfer auf 14B")]):
    ax.scatter(T[x], T[y], c=col, s=40, edgecolor="k", linewidth=0.3)
    ax.set_xlabel(x); ax.set_ylabel(y); ax.set_title(t, fontsize=10)
fig.suptitle("grün = Top 16, rot = Bottom 16"); fig.tight_layout(); fig.savefig(FIGS / "korrelat1_streu.png", dpi=150); plt.show()

In [ ]:
# Ergebnisse dieser Sitzung einpacken und herunterladen
import zipfile
zip_path = ROOT.parent / ("dsse_icl_vs_ft_gesamt.zip" if ANALYSIS_ONLY else "dsse_icl_vs_ft_results.zip" if N_SHARDS == 1
                         else f"dsse_icl_vs_ft_shard{SHARD_ID}.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(RUNS.glob("*.json")) + sorted(FIGS.glob("*.png")) + sorted(ROOT.glob("*.csv")):
        zf.write(p, p.relative_to(ROOT))
print("Ergebnisse:", zip_path)
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass
if USE_DRIVE:
    drive.flush_and_unmount()   # stellt sicher, dass alles in Drive angekommen ist
    print("Drive synchronisiert.")